# Human calibration of the VLM judge

**For both annotators:** open this notebook, set your name and the two paths below, and label each image. Your decisions are saved to `annotations_<your_name>.csv` after every image — so you can stop and resume anytime.

**The two annotators must label the same images.** Don't change the random seed in `exp3_sample_calibration.py`; the resulting manifest is what guarantees both of you see the same 360 images.

**Time estimate:** ~360 images × ~10 seconds each = ~1 hour.

## 1. Setup — edit these three values

- `ANNOTATOR_NAME`: your name or initials. Determines the output filename.
- `CALIBRATION_DIR`: where `calibration_manifest.csv` lives.
- `IMAGES_ROOT`: where the actual PNGs live (usually `data/eval_images/`).

The notebook reads images from `IMAGES_ROOT` directly, so you don't need to copy them — as long as you have read access to that folder.

In [1]:
ANNOTATOR_NAME = "Monica"                  
CALIBRATION_DIR = "data/calibration"       
IMAGES_ROOT = "data/eval_images"           

import csv
import os
from pathlib import Path

_here = Path().resolve()
if _here.name == "notebooks":
    os.chdir(_here.parent)

CALIBRATION_DIR = Path(CALIBRATION_DIR)
IMAGES_ROOT     = Path(IMAGES_ROOT)
MANIFEST_PATH = CALIBRATION_DIR / "calibration_manifest.csv"
OUT_PATH      = CALIBRATION_DIR / f"annotations_{ANNOTATOR_NAME}.csv"

assert MANIFEST_PATH.exists(), f'Manifest not found: {MANIFEST_PATH}'
assert IMAGES_ROOT.exists(),   f'Images root not found: {IMAGES_ROOT}'
print(f'Annotations will be saved to: {OUT_PATH}')
print(f'Images will be read from:     {IMAGES_ROOT}')

Annotations will be saved to: data\calibration\annotations_Monica.csv
Images will be read from:     data\eval_images


## 2. Load manifest and skip already-labeled rows

If you've labeled some before and are coming back, the notebook resumes from where you stopped.

In [2]:
with MANIFEST_PATH.open(newline="", encoding="utf-8") as f:
    all_rows = list(csv.DictReader(f))

already_labeled = set()
if OUT_PATH.exists():
    with OUT_PATH.open(newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            already_labeled.add((row["object"], row["color"], row["seed"]))

to_label = [r for r in all_rows if (r["object"], r["color"], r["seed"]) not in already_labeled]
print(f'Total images: {len(all_rows)}')
print(f'Already labeled: {len(already_labeled)}')
print(f'Remaining: {len(to_label)}')

Total images: 360
Already labeled: 0
Remaining: 360


## 3. Annotation interface

For each image, you'll see:
- The prompt the model was asked to draw (e.g., "a photo of a pink chalkboard")
- The image itself
- Four buttons covering the 2×2 outcomes (object correct? × color correct?)

**Calibration of your judgment:**
- *Object correct* = the main subject of the image is clearly the named object. Stylized/cartoon counts as correct if recognizable.
- *Color correct* = the color of **that object** matches the prompt color. If the object is wrong, color is automatically wrong too (just click "both wrong").
- When the image is ambiguous (e.g., two objects, partial color), use your best judgment — that's what kappa measures.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output, Image

out_f = OUT_PATH.open('a', newline='', encoding='utf-8')
writer = csv.DictWriter(out_f, fieldnames=[
    'object', 'color', 'seed', 'prompt', 'path',
    'object_correct', 'color_correct', 'annotator',
])
if OUT_PATH.stat().st_size == 0:
    writer.writeheader()
    out_f.flush()

state = {
    'queue': list(to_label),
    'index': 0,
    'current_row': None,
}

output = widgets.Output()

def show_current():
    """Render the current image and prompt."""
    with output:
        clear_output(wait=True)
        if not state['queue']:
            print('🎉 All images labeled. You can close the notebook.')
            out_f.close()
            return
        row = state['queue'][0]
        state['current_row'] = row
        done = len(all_rows) - len(state['queue'])
        total = len(all_rows)
        print(f'Progress: {done}/{total}  ({done/total:.0%})')
        print(f"Prompt asked for: {row['prompt']}")
        print(f"  expected object: {row['object']!r}, expected color: {row['color']!r}")
       
        img_path = IMAGES_ROOT / row['path']
        if not img_path.exists():
            print(f'⚠️  Image not found at {img_path}')
            print('   Check that IMAGES_ROOT in cell 1 points to your eval_images/ folder.')
            return
        display(Image(filename=str(img_path), width=384))

def record(object_correct: bool, color_correct: bool):
    """Save current label and advance."""
    row = state['current_row']
    if row is None:
        return
    writer.writerow({
        'object':  row['object'],
        'color':   row['color'],
        'seed':    row['seed'],
        'prompt':  row['prompt'],
        'path':    row['path'],
        'object_correct': int(object_correct),
        'color_correct':  int(color_correct),
        'annotator': ANNOTATOR_NAME,
    })
    out_f.flush()
    state['queue'].pop(0)
    show_current()

btn_both_ok = widgets.Button(description='✓ both correct',     button_style='success')
btn_obj_ok  = widgets.Button(description='object yes, color no', button_style='warning')
btn_col_ok  = widgets.Button(description='object no, color yes', button_style='warning')
btn_both_no = widgets.Button(description='✗ both wrong',       button_style='danger')

btn_both_ok.on_click(lambda _: record(True,  True))
btn_obj_ok.on_click( lambda _: record(True,  False))
btn_col_ok.on_click( lambda _: record(False, True))
btn_both_no.on_click(lambda _: record(False, False))

buttons = widgets.HBox([btn_both_ok, btn_obj_ok, btn_col_ok, btn_both_no])

display(buttons, output)
show_current()

Output()

## 4. Done

When the cell above shows "All images labeled", you're done. Send your `annotations_<your_name>.csv` to whoever is running the analysis. Don't worry about the order of rows — the analysis script matches by `(object, color, seed)`.